In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from IPython.display import clear_output, display
from tqdm.auto import tqdm
from argparse import Namespace

from ncpu.dataset import NCPUDataset, sample_4bit_adder
from ncpu.nca import NeuralCA

In [ ]:
device = "cuda"
ds_config = Namespace(
    W=117,
    H=117,
    r=4,
    spacing=(2, 50),
    sampler=sample_4bit_adder,
    balanced=False,
)

dataset = NCPUDataset(ds_config)

In [ ]:
nca_config = Namespace(
    channels=8,
    hidden_channels=128,
    fire_rate=0.99,
    alive_threshold=0.1,
    zero_initialization=False,
    mass_conserving="no",
    visual_channels=1,
    kernel_size=5,
    num_perception_kernels=5,
)

nca = NeuralCA(**vars(nca_config)).to(device)

In [ ]:
from ncpu.trainer import NCPUTrainer

optim_config = Namespace(
    lr=0.00002,
    batch_size=8,
    gaussian_noise=0.01,
)

trainer = NCPUTrainer(
    nca,
    dataset.get_dataloader(batch_size=optim_config.batch_size),
    lr=optim_config.lr,
    gaussian_noise=optim_config.gaussian_noise,
)
trainer.sanity_check()

In [ ]:
with torch.no_grad():
    info = trainer.optim_step(steps=(30, 80))
    trainer.display_optim_step(info, display_size=117, to_show=8)
    trainer.save_checkpoint()

In [ ]:
pbar = tqdm(range(100_000))

for i in pbar:
    info = trainer.optim_step(steps=(30, 80))
    loss = info["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    if i % 250 == 0:
        clear_output(wait=False)
        display(pbar.container)

        trainer.display_optim_step(info, display_size=100, to_show=8)
        trainer.save_checkpoint()